# Lectura de un CSV exportado desde una base de datos

Este ejemplo usa un archivo dummy, pero reproduce tres problemas comunes de dumps reales:

- una columna `varchar(12)` trae códigos opacos con comas internas sin escapado;
- el archivo no esta en UTF-8, sino en Windows-1252;
- algunos campos tienen saltos de linea embebidos.

La idea es leerlo primero de la forma mas ingenua posible y despues ir corrigiendo cada problema hasta obtener una tabla consistente.

In [1]:
from pathlib import Path
import csv
import pandas as pd

raw_dir = Path("data/raw")
processed_dir = Path("data/processed")

if not raw_dir.exists():
    raw_dir = Path("etl-03/data/raw")
    processed_dir = Path("etl-03/data/processed")

archivo = raw_dir / "producto_dummy_problemas.csv"
archivo

PosixPath('data/raw/producto_dummy_problemas.csv')

## 1. Primer intento: `pd.read_csv` con valores por defecto

Pandas asume UTF-8 por defecto. Si el archivo viene de un sistema Windows o de una base antigua, puede aparecer el primer problema antes incluso de mirar las columnas.

In [2]:
try:
    df = pd.read_csv(archivo)
except Exception as exc:
    print(type(exc).__name__)
    print(exc)

UnicodeDecodeError
'utf-8' codec can't decode byte 0xe9 in position 67: invalid continuation byte


## 2. Detectar el encoding por prueba

No siempre sabemos con que encoding fue exportado el archivo. Una estrategia practica es probar una lista de encodings frecuentes y mirar cuales pueden decodificar el archivo sin errores. Esto no garantiza automaticamente que sea el correcto, pero achica mucho el diagnostico.


In [3]:
encodings_candidatos = ["utf-8", "utf-8-sig", "cp1252", "latin1", "iso-8859-1"]

for encoding in encodings_candidatos:
    try:
        muestra = archivo.read_text(encoding=encoding)[:120]
    except UnicodeDecodeError as exc:
        print(f"{encoding:10} -> ERROR: {exc.reason}")
    else:
        print(f"{encoding:10} -> OK    : {muestra!r}")


utf-8      -> ERROR: invalid continuation byte
utf-8-sig  -> ERROR: invalid continuation byte
cp1252     -> OK    : 'CodigoProducto,Nombre,Categoria,Precio,Notas,Activo\n8:#(,yo!c0,Café molido clásico,Almacén,2450.50,"Paquete de 500 g\nOri'
latin1     -> OK    : 'CodigoProducto,Nombre,Categoria,Precio,Notas,Activo\n8:#(,yo!c0,Café molido clásico,Almacén,2450.50,"Paquete de 500 g\nOri'
iso-8859-1 -> OK    : 'CodigoProducto,Nombre,Categoria,Precio,Notas,Activo\n8:#(,yo!c0,Café molido clásico,Almacén,2450.50,"Paquete de 500 g\nOri'


En este ejemplo `latin1` tambien puede decodificar el archivo porque acepta cualquier byte, pero eso no significa que sea semanticamente correcto. Para exports de sistemas Windows en espanol, `cp1252` suele ser una mejor hipotesis que `latin1` porque preserva caracteres imprimibles tipicos de Windows.


## 3. Corregimos el encoding

Probamos con `cp1252`, un encoding muy comun en exports de sistemas Windows en espanol. Ahora pandas puede decodificar los caracteres, pero aparece el problema estructural: hay comas dentro de una columna que no fueron escapadas.

In [4]:
try:
    df = pd.read_csv(archivo, encoding="cp1252")
except Exception as exc:
    print(type(exc).__name__)
    print(exc)

ParserError
Error tokenizing data. C error: Expected 7 fields in line 4, saw 8



Si forzamos a pandas a saltar lineas problematicas, no arreglamos el dato: solamente escondemos filas. Esto sirve para diagnostico, no para una carga ETL confiable.

In [5]:
df_parcial = pd.read_csv(
    archivo,
    encoding="cp1252",
    on_bad_lines="skip",
)

df_parcial

,CodigoProducto,Nombre,Categoria,Precio,Notas,Activo
8:#(,yo!c0,Café molido clásico,Almacén,2450.50,Paquete de 500 g\r\nOrigen: Brasil. Tostado me...,1
K7?,x9!!c0,Té negro en hebras,Infusiones,1320.00,Blend intenso con aroma a bergamota.,1
Z0%/,pq7!A,Aceite de oliva extra virgen,Almacén,5200.00,"Botella de vidrio. Acidez menor a 0,5%.",1
R8&,55#zP1,Galletas con chips de chocolate,Almacén,1750.25,Caja exhibidora x 12 unidades.\r\nPuede conten...,0


## 4. Inspeccionamos como queda tokenizada cada fila

El modulo `csv` de Python si respeta los campos con saltos de linea cuando estan entre comillas. Eso nos permite distinguir dos cosas:

- los saltos de linea embebidos no rompen la fila si el campo esta bien citado;
- la primera columna sigue rota porque contiene comas sin comillas.

In [6]:
with archivo.open("r", encoding="cp1252", newline="") as f:
    filas = list(csv.reader(f))

for i, fila in enumerate(filas):
    print(i, len(fila), fila)

0 6 ['CodigoProducto', 'Nombre', 'Categoria', 'Precio', 'Notas', 'Activo']
1 7 ['8:#(', 'yo!c0', 'Café molido clásico', 'Almacén', '2450.50', 'Paquete de 500 g\r\nOrigen: Brasil. Tostado medio.', '1']
2 7 ['K7?', 'x9!!c0', 'Té negro en hebras', 'Infusiones', '1320.00', 'Blend intenso con aroma a bergamota.', '1']
3 8 ['M2$', 'ab@', '90', 'Jabón líquido para manos', 'Limpieza', '890.75', 'Fragancia limón.\r\nApto dispenser.', '1']
4 7 ['Z0%/', 'pq7!A', 'Aceite de oliva extra virgen', 'Almacén', '5200.00', 'Botella de vidrio. Acidez menor a 0,5%.', '1']
5 7 ['R8&', '55#zP1', 'Galletas con chips de chocolate', 'Almacén', '1750.25', 'Caja exhibidora x 12 unidades.\r\nPuede contener maní.', '0']
6 8 ['9V@', '3D?', '02', 'Yerba mate suave', 'Infusiones', '3100.00', 'Producto estacionado. Elaboración argentina.', '1']


La cabecera tiene 6 columnas. Las filas con mas de 6 valores tienen tokens extra al principio: son partes del codigo opaco `CodigoProducto`, no columnas nuevas.

In [7]:
cabecera = filas[0]
esperadas = len(cabecera)

print(cabecera)
print("Columnas esperadas:", esperadas)

for i, fila in enumerate(filas[1:], start=1):
    print(f"fila {i}: {len(fila)} campos")

['CodigoProducto', 'Nombre', 'Categoria', 'Precio', 'Notas', 'Activo']
Columnas esperadas: 6
fila 1: 7 campos
fila 2: 7 campos
fila 3: 8 campos
fila 4: 7 campos
fila 5: 7 campos
fila 6: 8 campos


## 5. Reparacion controlada

Como conocemos el contrato del archivo, sabemos que la tabla final debe tener 6 columnas. Si una fila trae mas campos, el excedente pertenece a la primera columna: el codigo opaco venia de un `varchar(12)` y puede contener comas. Reunimos esos tokens con comas y dejamos el resto de la fila intacta.

In [8]:
def reparar_fila(fila, cantidad_columnas):
    if len(fila) < cantidad_columnas:
        raise ValueError(f"Fila con menos campos de los esperados: {fila}")

    extras = len(fila) - cantidad_columnas
    codigo = ",".join(fila[: extras + 1])
    resto = fila[extras + 1 :]
    return [codigo, *resto]

filas_reparadas = [reparar_fila(fila, esperadas) for fila in filas[1:]]

df = pd.DataFrame(filas_reparadas, columns=cabecera)
df

,CodigoProducto,Nombre,Categoria,Precio,Notas,Activo
0,"8:#(,yo!c0",Café molido clásico,Almacén,2450.50,Paquete de 500 g\r\nOrigen: Brasil. Tostado me...,1
1,"K7?,x9!!c0",Té negro en hebras,Infusiones,1320.00,Blend intenso con aroma a bergamota.,1
2,"M2$,ab@,90",Jabón líquido para manos,Limpieza,890.75,Fragancia limón.\r\nApto dispenser.,1
3,"Z0%/,pq7!A",Aceite de oliva extra virgen,Almacén,5200.00,"Botella de vidrio. Acidez menor a 0,5%.",1
4,"R8&,55#zP1",Galletas con chips de chocolate,Almacén,1750.25,Caja exhibidora x 12 unidades.\r\nPuede conten...,0
5,"9V@,3D?,02",Yerba mate suave,Infusiones,3100.00,Producto estacionado. Elaboración argentina.,1


## 6. Tipos de datos y limpieza minima

Una vez que la estructura es correcta, recien ahi conviene convertir tipos y decidir que hacer con los saltos de linea embebidos. En una carga real, a veces se preservan; para analisis tabular muchas veces se normalizan a espacios.

In [9]:
df_limpio = df.copy()

df_limpio["Precio"] = pd.to_numeric(df_limpio["Precio"])
df_limpio["Activo"] = df_limpio["Activo"].map({"1": True, "0": False})
df_limpio["Notas"] = df_limpio["Notas"].str.replace(r"\s*\r?\n\s*", " | ", regex=True)
df_limpio["CodigoProductoHex"] = df_limpio["CodigoProducto"].map(
    lambda valor: valor.encode("cp1252").hex(" ")
)

df_limpio

,CodigoProducto,Nombre,Categoria,Precio,Notas,Activo,CodigoProductoHex
0,"8:#(,yo!c0",Café molido clásico,Almacén,2450.50,Paquete de 500 g | Origen: Brasil. Tostado medio.,True,38 3a 23 28 2c 79 6f 21 63 30
1,"K7?,x9!!c0",Té negro en hebras,Infusiones,1320.00,Blend intenso con aroma a bergamota.,True,4b 37 3f 2c 78 39 21 21 63 30
2,"M2$,ab@,90",Jabón líquido para manos,Limpieza,890.75,Fragancia limón. | Apto dispenser.,True,4d 32 24 2c 61 62 40 2c 39 30
3,"Z0%/,pq7!A",Aceite de oliva extra virgen,Almacén,5200.00,"Botella de vidrio. Acidez menor a 0,5%.",True,5a 30 25 2f 2c 70 71 37 21 41
4,"R8&,55#zP1",Galletas con chips de chocolate,Almacén,1750.25,Caja exhibidora x 12 unidades. | Puede contene...,False,52 38 26 2c 35 35 23 7a 50 31
5,"9V@,3D?,02",Yerba mate suave,Infusiones,3100.00,Producto estacionado. Elaboración argentina.,True,39 56 40 2c 33 44 3f 2c 30 32


## 7. Validaciones simples para ETL

Antes de cargar el dato, dejamos algunas comprobaciones basicas: cantidad de columnas, nulos y tipos finales.

In [10]:
assert list(df_limpio.columns[:esperadas]) == cabecera
assert "CodigoProductoHex" in df_limpio.columns
assert not df_limpio["CodigoProducto"].isna().any()
assert not df_limpio["CodigoProductoHex"].isna().any()
assert not df_limpio["Nombre"].isna().any()

df_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CodigoProducto     6 non-null      str    
 1   Nombre             6 non-null      str    
 2   Categoria          6 non-null      str    
 3   Precio             6 non-null      float64
 4   Notas              6 non-null      str    
 5   Activo             6 non-null      bool   
 6   CodigoProductoHex  6 non-null      str    
dtypes: bool(1), float64(1), str(5)
memory usage: 1.1 KB


## 7. Export opcional ya saneado

A partir de aca guardamos un CSV normalizado en UTF-8 dentro de `data/processed`. Este archivo ya no deberia necesitar reglas especiales para ser leido.

In [11]:
processed_dir.mkdir(parents=True, exist_ok=True)
salida = processed_dir / "producto_dummy_limpio.csv"
df_limpio.to_csv(salida, index=False, encoding="utf-8")

pd.read_csv(salida)

,CodigoProducto,Nombre,Categoria,Precio,Notas,Activo,CodigoProductoHex
0,"8:#(,yo!c0",Café molido clásico,Almacén,2450.50,Paquete de 500 g | Origen: Brasil. Tostado medio.,True,38 3a 23 28 2c 79 6f 21 63 30
1,"K7?,x9!!c0",Té negro en hebras,Infusiones,1320.00,Blend intenso con aroma a bergamota.,True,4b 37 3f 2c 78 39 21 21 63 30
2,"M2$,ab@,90",Jabón líquido para manos,Limpieza,890.75,Fragancia limón. | Apto dispenser.,True,4d 32 24 2c 61 62 40 2c 39 30
3,"Z0%/,pq7!A",Aceite de oliva extra virgen,Almacén,5200.00,"Botella de vidrio. Acidez menor a 0,5%.",True,5a 30 25 2f 2c 70 71 37 21 41
4,"R8&,55#zP1",Galletas con chips de chocolate,Almacén,1750.25,Caja exhibidora x 12 unidades. | Puede contene...,False,52 38 26 2c 35 35 23 7a 50 31
5,"9V@,3D?,02",Yerba mate suave,Infusiones,3100.00,Producto estacionado. Elaboración argentina.,True,39 56 40 2c 33 44 3f 2c 30 32


## 10. Discusion: reemplazar o conservar el codigo original

En este notebook dejamos dos columnas: `CodigoProducto`, con el valor original reconstruido, y `CodigoProductoHex`, con una representacion mas segura para transportar o comparar.

Para una clase, conservar ambas columnas es util porque permite auditar la transformacion: se ve el dato problematico y, al lado, su version hexadecimal. Tambien ayuda a explicar que el campo venia de un `varchar(12)`, pero no necesariamente era un identificador pensado para ser leido por personas.

En una carga productiva, en cambio, puede convenir que la clave usable sea directamente la version hexadecimal. En ese caso, una alternativa clara seria renombrar el valor crudo como `CodigoProductoOriginal` y usar `CodigoProducto` para el identificador saneado.

Ventajas de reemplazar `CodigoProducto` por hex:

- evita comas, simbolos raros y problemas de encoding en pasos posteriores;
- deja una clave estable para CSV, logs, bases de datos o APIs;
- comunica que el valor debe tratarse como bytes opacos, no como texto de negocio.

Desventajas de reemplazarlo:

- se pierde legibilidad inmediata frente al archivo original;
- para auditar contra el dump fuente hay que convertir de vuelta o conservar otra columna;
- puede ocultar el problema pedagogico si el objetivo es mostrar por que fallaba la lectura.

Por eso, una regla practica podria ser: durante exploracion, conservar ambas columnas; para una tabla final de consumo, usar una sola clave limpia y guardar el original solo si hace falta trazabilidad.